In [ ]:
# 🎯 1. Project Idea (Concept)

# You have:

# movies.csv → movie info (title, genres)
# ratings.csv → user ratings

# 👉 Goal:
# Group movies (or users) into clusters → then recommend similar movies

# 👉 Example:

# Cluster 1 = Action movies lovers
# Cluster 2 = Romantic movies lovers

In [ ]:
# 📊 2. Strategy (Important)

# You have 2 main ways:

# ✅ Option A (Recommended for you)

# 👉 Cluster movies based on user ratings

# ✅ Option B

# 👉 Cluster users based on behavior

# 🔥 3. Steps Overview
# Load data
# Merge datasets
# Create matrix (pivot)
# Fill missing values
# Apply K-Means
# Recommend movies

In [13]:
# 🧠 4. Implementation (Code + Explain Khmer)
# Step 1: Import libraries
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [47]:
# Step 2: Load dataset
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

dddd = movies["movieId"].tolist()[:10]
for x in dddd:
    title =movies.loc[movies["movieId"]==x,"title"]
    print(title.values[0])

Toy Story (1995)
Jumanji (1995)
Grumpier Old Men (1995)
Waiting to Exhale (1995)
Father of the Bride Part II (1995)
Heat (1995)
Sabrina (1995)
Tom and Huck (1995)
Sudden Death (1995)
GoldenEye (1995)


In [21]:
# Filter high rating only (SMART)
top_data = ratings[ratings['rating'] >= 3.0].head(100000)

In [22]:
# Step 3: Merge data
data = pd.merge(top_data, movies, on="movieId")
data = data.head(100000)
data.info()
# 👉 Khmer:

# ផ្សំ ratings + movies
# ដើម្បីយើងបាន user + movie + rating + title

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100000 non-null  int64  
 1   movieId    100000 non-null  int64  
 2   rating     100000 non-null  float64
 3   timestamp  100000 non-null  int64  
 4   title      100000 non-null  object 
 5   genres     100000 non-null  object 
dtypes: float64(1), int64(3), object(2)
memory usage: 4.6+ MB


In [23]:
data.head()

,userId,movieId,rating,timestamp,title,genres
0,1,296,5.0,1147880044,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
1,1,306,3.5,1147868817,Three Colors: Red (Trois couleurs: Rouge) (1994),Drama
2,1,307,5.0,1147868828,Three Colors: Blue (Trois couleurs: Bleu) (1993),Drama
3,1,665,5.0,1147878820,Underground (1995),Comedy|Drama|War
4,1,899,3.5,1147868510,Singin' in the Rain (1952),Comedy|Musical|Romance


In [24]:
# Step 4: Create Pivot Table (IMPORTANT)
movie_matrix = data.pivot_table(index='title', columns='userId', values='rating')

# 👉 Khmer:

# row = movie
# column = user
# value = rating

# 👉 Result:

#         user1 user2 user3
# Movie A   5     4    NaN
# Movie B   3    NaN   4
movie_matrix

userId,1,2,3,4,5,6,7,8,9,10,...,855,856,857,858,859,860,861,862,863,864
title,,,,,,,,,,,,,,,,,,,,,
$5 a Day (2008),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Round Midnight (1986),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Salem's Lot (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Til There Was You (1997),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"'burbs, The (1989)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Когда зажигаются ёлки (1950),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Обезьянки и грабители (1985),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Он вам не Димон (2017),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
# Step 5: Fill missing values
movie_matrix = movie_matrix.fillna(0)

# 👉 Khmer:

# NaN = user មិនបាន rate
# ដាក់ 0 ជំនួស
movie_matrix

userId,1,2,3,4,5,6,7,8,9,10,...,855,856,857,858,859,860,861,862,863,864
title,,,,,,,,,,,,,,,,,,,,,
$5 a Day (2008),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Round Midnight (1986),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Salem's Lot (2004),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Til There Was You (1997),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"'burbs, The (1989)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Когда зажигаются ёлки (1950),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Обезьянки и грабители (1985),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Он вам не Димон (2017),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [26]:
movie_matrix.to_excel("new-data.xlsx", index=False)

In [29]:
movie_matrix.isnull().count()

userId
1      10075
2      10075
3      10075
4      10075
5      10075
       ...  
860    10075
861    10075
862    10075
863    10075
864    10075
Length: 864, dtype: int64

In [30]:
# Step 6: Scaling (optional but good)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(movie_matrix)

# 👉 Khmer:
# បម្លែង data អោយមាន scale តូចស្មើគ្នា
# Step 7: Apply K-Means
kmeans = KMeans(n_clusters=5, random_state=42)
kmeans.fit(scaled_data)

movie_matrix['Cluster'] = kmeans.labels_

# 👉 Khmer:

# បែងចែក movie ទៅជា 5 groups

In [31]:
# 🎬 5. Recommendation Function
def recommend_movies(movie_name):
    cluster = movie_matrix.loc[movie_name, 'Cluster']
    similar_movies = movie_matrix[movie_matrix['Cluster'] == cluster]
    
    return similar_movies.index.tolist()[:10]

# 👉 Khmer:

# រក cluster របស់ movie
# យក movie ដែលនៅ cluster ដូចគ្នា

In [32]:
# Example:
recommend_movies("Toy Story (1995)")

# 👉 Output:

# ["Toy Story 2", "A Bug's Life", "Monsters Inc", ...]

['Birdcage, The (1996)',
 'Broken Arrow (1996)',
 'Dead Man Walking (1995)',
 'Dragonheart (1996)',
 'Eraser (1996)',
 'Executive Decision (1996)',
 'Fargo (1996)',
 'Father of the Bride Part II (1995)',
 'Grumpier Old Men (1995)',
 'Happy Gilmore (1996)']

In [ ]:
# 📈 6. Visualization (Optional)
import matplotlib.pyplot as plt

plt.scatter(movie_matrix.iloc[:,0], movie_matrix.iloc[:,1], c=kmeans.labels_)
plt.show()